# Embedding Clustering

Cluster both template Jina embeddings and title-only Jina embeddings. Use K-Means when every product should be assigned to a group, and HDBSCAN when noise/outlier detection is useful.

In [15]:
from sklearn.cluster import HDBSCAN, KMeans
from sklearn.preprocessing import normalize


def create_clusters(embeddings, min_cluster_size=20):
    clusterer = HDBSCAN(min_cluster_size=min_cluster_size, metric="euclidean", copy=False)
    labels = clusterer.fit_predict(embeddings)
    return labels


def create_kmeans_clusters(embeddings, n_clusters=20, random_state=42, normalize_embeddings=True):
    if normalize_embeddings:
        embeddings = normalize(embeddings)

    clusterer = KMeans(n_clusters=n_clusters, random_state=random_state, n_init="auto")
    labels = clusterer.fit_predict(embeddings)
    return labels

In [16]:
from collections import Counter
from pathlib import Path

import numpy as np
from datasets import load_dataset


N_CLUSTERS = 20


def resolve_path(path):
    path = Path(path)
    if path.exists():
        return path
    return Path("ecom") / path


def load_embedding_matrix(path, column_name):
    dataset = load_dataset("parquet", data_files=str(resolve_path(path)), split="train")
    return np.asarray(dataset[column_name], dtype=np.float32)


template_embeddings = load_embedding_matrix(
    "data/embeddings/embeddings_template_jina-embeddings-v5-text-small.parquet",
    "embedding_doc_embedding",
)
title_embeddings = load_embedding_matrix(
    "data/embeddings/embeddings_title_jina-embeddings-v5-text-small.parquet",
    "title_embedding",
)

template_labels = create_kmeans_clusters(template_embeddings, n_clusters=N_CLUSTERS)
title_labels = create_kmeans_clusters(title_embeddings, n_clusters=N_CLUSTERS)

template_counts = Counter(template_labels.tolist())
title_counts = Counter(title_labels.tolist())


def print_cluster_summary(name, embeddings, labels, counts):
    print(name)
    print(f"rows: {embeddings.shape[0]}")
    print(f"embedding_dim: {embeddings.shape[1]}")
    print(f"clusters_found: {len(counts)}")
    print("unassigned_points: 0")
    print("top_cluster_sizes:", sorted(counts.values(), reverse=True)[:10])
    print("first_20_labels:", labels[:20].tolist())


print_cluster_summary("Template embeddings", template_embeddings, template_labels, template_counts)
print()
print_cluster_summary("Title-only embeddings", title_embeddings, title_labels, title_counts)

Template embeddings
rows: 10000
embedding_dim: 1024
clusters_found: 20
unassigned_points: 0
top_cluster_sizes: [1170, 873, 827, 774, 719, 714, 659, 528, 504, 466]
first_20_labels: [1, 0, 9, 19, 11, 13, 12, 10, 2, 1, 10, 2, 13, 15, 10, 11, 18, 3, 14, 7]

Title-only embeddings
rows: 10000
embedding_dim: 1024
clusters_found: 20
unassigned_points: 0
top_cluster_sizes: [1145, 797, 754, 705, 695, 583, 550, 526, 523, 442]
first_20_labels: [8, 0, 14, 9, 19, 2, 7, 17, 17, 8, 15, 5, 12, 6, 15, 19, 18, 3, 6, 17]


In [17]:
raw_file = resolve_path("data/sampled/sample.raw.10k.parquet")
raw_dataset = load_dataset("parquet", data_files=str(raw_file), split="train")

clustered = raw_dataset.add_column("template_cluster", template_labels.tolist())
clustered = clustered.add_column("title_cluster", title_labels.tolist())
clustered_df = clustered.to_pandas()

clustered_df[
    ["template_cluster", "title_cluster", "main_category", "title", "average_rating", "rating_number", "price"]
].head()

,template_cluster,title_cluster,main_category,title,average_rating,rating_number,price
0,1,8,All Beauty,Skinny Black Narrow Headband in Patent Leather,5.0,1,None
1,0,0,All Beauty,Swan makeup brush set 11pcs eye shadow brush c...,2.2,3,None
2,9,14,All Beauty,Feret Parfumeur L’Eau de Monsieur Eau de Parfu...,4.5,4,58.5
3,19,9,All Beauty,"Magnetic Eyelashes Kits Set, 2019 Minso Natura...",2.5,19,None
4,11,19,All Beauty,RazoRock 400 Plissoft Synthetic Shaving Brush ...,4.7,4,26.95


In [18]:
cluster_source = "template_cluster"  # change to "title_cluster" to inspect title-only clusters
cluster_id = 0

clustered_df[clustered_df[cluster_source] == cluster_id][
    ["template_cluster", "title_cluster", "main_category", "title", "average_rating", "rating_number", "price"]
].head(20)

,template_cluster,title_cluster,main_category,title,average_rating,rating_number,price
1,0,0,All Beauty,Swan makeup brush set 11pcs eye shadow brush c...,2.2,3,None
52,0,0,All Beauty,Magic Stylo Semi Permanent Makeup Pen (Black V...,3.9,9,None
79,0,0,All Beauty,Absolute New York Eyebrow Pencil (CHARCOAL GREY),4.1,62,None
109,0,16,All Beauty,Deluxe 2.0 Battery - Aeroblend Airbrush Makeup...,5.0,4,None
185,0,0,All Beauty,FONY Mermaid Makeup Brushes 10 Pcs Professiona...,3.5,3,None
220,0,0,All Beauty,LHEI 7Pcs Mermaid colorful Rainbow Makeup Brus...,3.9,10,None
238,0,0,All Beauty,[Lioele] Auto Eyebrow 2EA / Dual function eyeb...,5.0,2,None
272,0,0,All Beauty,"Eye Brush Set, 20 pcs Unicorn Eyeshadow Eyelin...",3.9,25,None
286,0,0,All Beauty,"Eyeshadow Sticker 180 PCS,Professional Eyeshad...",4.1,11,None
307,0,0,All Beauty,One Step Eyebrow Stamp Shaping Kit Brow Powder...,3.0,4,None


In [19]:
def print_cluster_samples(cluster_column, counts, sample_size=5):
    print(f"\n{cluster_column}")
    for cluster_id in sorted(clustered_df[cluster_column].unique()):
        cluster_sample = clustered_df[clustered_df[cluster_column] == cluster_id].head(sample_size)
        print(f"\nCluster {cluster_id} | size={counts[cluster_id]}")
        for title in cluster_sample["title"]:
            print(f"- {title}")


print_cluster_samples("template_cluster", template_counts)
print_cluster_samples("title_cluster", title_counts)


template_cluster

Cluster 0 | size=403
- Swan makeup brush set 11pcs eye shadow brush concealer professional makeup brush
- Magic Stylo Semi Permanent Makeup Pen (Black Velvet)
- Absolute New York Eyebrow Pencil (CHARCOAL GREY)
- Deluxe 2.0 Battery - Aeroblend Airbrush Makeup PRO Starter Kit - Professional Cosmetic System - 24 Color
- FONY Mermaid Makeup Brushes 10 Pcs Professional Makeup Brush Set, Premium Silky Soft Synthetic Bristles with Golden Mermaid Handle Cosmetics Brush Kit (10,Golden)

Cluster 1 | size=528
- Skinny Black Narrow Headband in Patent Leather
- Hipsy Adjustable Non Slip Cute Fashion Bling Glitter Hair Headbands for Women Girls & Teens 2-Pack (Wave Gunmetal & Black)
- 3 Pieces Soft Sleep Cap – Night Satin Bonnet with Wide Premium Elastic Band for Women
- Relbcy Boho Headband Flower Hair Bands Knoted Turban Headbands Stretch Head Wraps Cloth Head Bands for Women and Girls (Type D)
- Premium Quality Satin Sleeping Cap,1 Pack Wide Band Night Cap Satin Bonnet for Natu